In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q /content/drive/MyDrive/coral_condition/coral_images_dry.zip -d /content/imagens_extraidas

In [ ]:
!pip install ultralytics

In [ ]:
import pandas as pd
import os
import shutil
import glob
from sklearn.model_selection import train_test_split

# Limpa a tentativa anterior para não dar conflito
## no caso aqui temos que lembrar o csv e imagens originais tem mais de uma
## característica associada e para isso criamos uma ordem de prioridade
shutil.rmtree('/content/dataset_yolo', ignore_errors=True)

# 1. Carregar os dados
annotations = pd.read_csv('/content/drive/MyDrive/coral_condition/annotations.csv')
annotations['label'] = annotations['label'].astype(str)

# 2. Definir a prioridade e extrair a label dominante
priority = ['6', '8', '7', '2', '5', '3', '4', '1', '0']

def get_dominant_label(label_str):
    labels = label_str.split(',')
    for p in priority:
        if p in labels:
            return p
    return '0'

annotations['dominant_label'] = annotations['label'].apply(get_dominant_label)

label_map = {
    '0': 'Background', '1': 'Saudavel', '2': 'Comprometido',
    '3': 'Morto', '4': 'Escombro', '5': 'Competicao',
    '6': 'Doenca', '7': 'Predacao', '8': 'Danos_Fisicos'
}

# 3. Separar Treino e Validação
## 80% treino, 20% validação - stratify - com isso se 10% de tudo é pred.
## o treino terá 10% pred e vali
train_df, val_df = train_test_split(
    annotations, test_size=0.2, random_state=42, stratify=annotations['dominant_label']
)

base_dir = '/content/dataset_yolo'

# 4. O Truque: Mapear todas as imagens extraídas dinamicamente
print("Mapeando o esconderijo das imagens...")
todas_imagens = glob.glob('/content/imagens_extraidas/**/*', recursive=True)
mapa_imagens = {}

for caminho in todas_imagens:
    if os.path.isfile(caminho):
        nome_arquivo = os.path.basename(caminho)
        nome_sem_ext, _ = os.path.splitext(nome_arquivo)
        mapa_imagens[nome_sem_ext] = caminho

print(f"Total de arquivos encontrados na extração: {len(mapa_imagens)}")

# 5. Criar pastas e copiar
for split_name, df_split in [('train', train_df), ('val', val_df)]:
    for folder_name in label_map.values():
        os.makedirs(os.path.join(base_dir, split_name, folder_name), exist_ok=True)

    print(f"Copiando imagens para a pasta {split_name}...")
    copiadas = 0
    for index, row in df_split.iterrows():
        patch_id = row['patchid']
        dominant_label = row['dominant_label']
        folder_name = label_map[dominant_label]

        # Pega o caminho real e a extensão correta direto do nosso mapa
        src_path = mapa_imagens.get(patch_id)

        if src_path:
            extensao = os.path.splitext(src_path)[1]
            dst_path = os.path.join(base_dir, split_name, folder_name, f"{patch_id}{extensao}")
            shutil.copy(src_path, dst_path)
            copiadas += 1

    print(f"Sucesso: {copiadas} imagens copiadas para {split_name}!")

print("Tudo pronto! Pode rodar o YOLO novamente.")

In [ ]:
from ultralytics import YOLO

# Carrega o modelo nano de classificação (já pré-treinado)
model = YOLO('yolo26n-cls.pt')

# Inicia o treinamento
resultados = model.train(
    data='/content/dataset_yolo', # Onde o script organizador guardou as pastas
    epochs=10,                    # Vamos começar com 10 épocas para um teste rápido
    imgsz=224,                    # Tamanho padrão de imagem para classificação
    device=0                      # Garante que ele vai usar a GPU T4 que você ativou
)

In [ ]:
from ultralytics import YOLO

# Carrega o modelo base novamente para um treino completo
model = YOLO('yolo26n-cls.pt')

# Inicia o treinamento com 30 épocas
resultados = model.train(
    data='/content/dataset_yolo',
    epochs=30,                    # Agora rumo aos 80%+ de acerto!
    imgsz=224,                    ##já que fiz um com 10 épocas e um com 30 épocas - train.2 e train.3
    device=0
)

In [ ]:
from ultralytics import YOLO
import glob
import os

# 1. Procura automaticamente o modelo mais recente que você acabou de treinar
lista_modelos = glob.glob('/content/runs/classify/*/weights/best.pt')
caminho_modelo = max(lista_modelos, key=os.path.getctime)
meu_modelo = YOLO(caminho_modelo)

# 2. As suas três fotos de teste do Google
imagens_teste = [
    'coral4.jpg',
    'coral3.jpg',
    'coral2.jpg',
    'coral1.jpg',
    'coral5.jpg',
    'coral6.jpg',
    'coral7.jpg'
]

print("Fazendo as previsões com o modelo treinado...\n")

# 3. Faz a mágica acontecer
for img in imagens_teste:
    if os.path.exists(img):
        resultado = meu_modelo(img)[0]
        classe_prevista = resultado.names[resultado.probs.top1]
        confianca = resultado.probs.top1conf * 100

        # Pega só o comecinho do nome pra não poluir a tela
        nome_curto = img[:20] + "..." if len(img) > 20 else img

        print(f"📸 Foto: {nome_curto}")
        print(f"🔍 Diagnóstico: {classe_prevista}")
        print(f"📊 Confiança: {confianca:.2f}%\n")
    else:
        print(f"❌ Ops, não achei a foto '{img}'. Você arrastou ela pros arquivos do Colab?")